# Readaloud alignment on a free GPU (Colab or Kaggle)

Step 2 of the readaloud toolkit: runs Qwen3-ForcedAligner on `bundle.zip` made by `prepare.py` (which already did the
transcript, VAD and chunk plan on your computer), and gives you back `results.zip` (`timings.json`) for `build.py`.

**Colab:** *Runtime → Change runtime type → T4 GPU*. Drag `bundle.zip` into the *Files* panel (left sidebar) and wait
until the upload has finished (the file grows in 4 MB steps while uploading; compare its size with the local file),
then *Runtime → Run all*. `results.zip` downloads automatically at the end
(it is also in the Files panel).

**Kaggle:** *Settings → Accelerator → GPU T4*, *Settings → Internet → On* (needs a phone-verified account).
Upload `bundle.zip` as a private Dataset (*Add Input → Upload*), then *Run All*. Download `results.zip` from the *Output* panel.

If step 3 is interrupted, just run it again: finished chunks are kept (for as long as the runtime lives).

In [ ]:
# 1. Install the aligner (keeps the platform's own GPU build of PyTorch)
# pip may print 'dependency conflicts' about diffusers / huggingface-hub: harmless, those packages aren't used
!pip install -q qwen-asr==0.0.6 soundfile
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable a GPU accelerator first")

In [ ]:
# 2. Find the bundle and unpack it into a writable work folder (already-aligned chunks are kept)
import glob, os, shutil, zipfile
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ    # (Colab also has a /kaggle folder)
WORK = "/kaggle/working/work" if ON_KAGGLE else "/content/work"

if ON_KAGGLE:
    zips = glob.glob("/kaggle/input/**/bundle.zip", recursive=True)
    found = glob.glob("/kaggle/input/**/book.json", recursive=True)
    if zips:
        zipfile.ZipFile(zips[0]).extractall(WORK)
    elif found:                                   # Kaggle unpacks uploaded zips automatically
        shutil.copytree(os.path.dirname(found[0]), WORK, dirs_exist_ok=True)
    else:
        raise SystemExit("No bundle found - add bundle.zip as an input dataset")
else:
    # drag bundle.zip into the Files panel (left sidebar), or put it in Google Drive and mount Drive there
    found = glob.glob("/content/bundle.zip") + glob.glob("/content/drive/MyDrive/**/bundle.zip", recursive=True)
    if not found:
        raise SystemExit("No bundle.zip in /content or Google Drive - upload it via the Files panel, wait until the upload has finished, then run this cell again")
    try:
        zipfile.ZipFile(found[0]).extractall(WORK)
    except zipfile.BadZipFile:
        raise SystemExit(f"{found[0]} is incomplete or damaged ({os.path.getsize(found[0]):,} bytes) - "
                         "if the upload is still running, wait for it; otherwise delete it and upload again")
print(sorted(os.listdir(WORK)))

In [ ]:
# 3. Precise alignment on the GPU in float16 (~8 min for a 15 h book on a T4; re-run this cell to resume after an interruption)
!cd {WORK} && python scripts/align.py . 2>&1 | grep --line-buffered -v -i warn

In [ ]:
# 4. Package the result
OUT = "/kaggle/working/results.zip" if ON_KAGGLE else "/content/results.zip"
with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(os.path.join(WORK, "timings.json"), "timings.json")
    z.write(os.path.join(WORK, "units.json"), "units.json")      # the aligner's raw output, for diagnosis
print("wrote", OUT, os.path.getsize(OUT) // 1024, "KB")
if not ON_KAGGLE:
    from google.colab import files
    files.download(OUT)